# Path Parameter and Query Parameters

In [ ]:
from fastapi import FastAPI
from enum import Enum

class ModelName(str, Enum):
    alexnet = "alexnet"
    resnet = "resnet"
    lenet = "lenet"

app = FastAPI()

# Basics of FASTAPI
@app.get('/')
async def root() -> dict[str, str]:
    return {"message": "Hello world!"}

# Path Parameters
@app.get('/items/{item_id}')
async def read_item(item_id: int) -> dict[str, int]:
    return {"item_id": item_id}

## Order Matters
@app.get('/users/me')
async def read_user_me() -> dict[str, str]:
    return {"user_id": "This is current user"}

@app.get('/users/{user_id}')
async def read_user(user_id: str) -> dict[str, str]:
    return {"user_id": user_id}

## Predefined Values
@app.get('/models/{model_name}')
async def read_model(model_name: ModelName):
    if model_name is ModelName.alexnet:
        return {"model_name": model_name, "message": "Let's go ALEX!"}
    if model_name.value == "lenet":
        return {"model_name": model_name, "message": "BOOYAH lenet! You're a machine!"}
    return {"model_name": model_name, "message": f"You're doing okay {model_name.value}"}

## Path Converter
@app.get('/files/{path:path}')
async def get_path(path: str) -> dict[str, str]:
    return {"path": path} 

# Query Parameters
## Defaults
fake_items_db = [{"item_name": "Foo"}, {"item_name": "Bar"}, {"item_name": "Baz"}]

@app.get('/items_list/')
async def read_query(skip: int = 0, limit: int = 10):
    return fake_items_db[skip : skip + limit]

## Optional Parameters having | None = None will turn any parameter into an optional one
## Type coversion from bool (i.e., inputing True, 1, true, on, yes)
@app.get('/queries/{query_id}')
async def get_query(query_id: str, q: str | None = None, short: bool = False) -> dict[str, str]:
    queries = {"queries": query_id}
    if q:
        queries.update({"question": q}) 
    if short:
        queries.update({"short": "Admin"})
    return queries

## Multiple path and queries
@app.get('/items_multiple/{item_id}/queries_multiple/{query_id}')
async def multiple_path(item_id: int, query_id: str, q: str | None = None, short: bool = False) -> dict[str, str | bool]:
    multiple_queries = {"item_id": item_id, "query_id": query_id}
    if q:
        multiple_queries.update({"question": q})
    if short:
        multiple_queries.update({"privilage": "Admin"})
    return multiple_queries
# So a correct path would be /items_multiple/15972134/queries_multiple/hoah35jhnb25?q=why?&short=on

## Required parameter
@app.get('/req_item/{item_id}')
async def req_item(item_id: int, needy: str) -> dict[str, int | str]:
    return {"item": item_id, "needs": needy}
# A correct path would be /req_item/129874?needy=dikil
# If there's no needy, then there will be error

# Request Body

In [1]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    description: str | None = None
    price: float
    tax: float | None = None

app = FastAPI()

@app.post("/items/")
async def create_item(item: Item):
    item_dict = item.model_dump()
    if item.tax is not None:
        pat = item.price + item.tax
        item_dict.update({"price_after_tax": pat})
    else:
        pat = item.price*0.11
        item_dict.update({"price_after_tax": pat})
    return item

@app.put("/items_query/{item_id}/")
async def create_item_query(item_id: str, item: Item, q: str | None = None):
    result = {"item_id": item_id, **item.model_dump()}
    if q:
        result.update({"question": q})
    return result
# Note: item.model_dump() is a function to convert item instance (from item: Item) 
# and turn it into a json. Using **item.model_dump() is to unpack the json into a dictionary.

# 